# Phân tích FP/FN: YOLOv8s và YOLOv8s-P2

Chạy **Run All** từ thư mục dự án hoặc `scripts/`. Notebook dùng test split và
threshold đã khóa, kiểm tra hash checkpoint, xuất bảng so sánh, gallery và báo cáo
vào `evaluation/analysis/`. Không thay đổi ngưỡng dựa trên test.

Cần cài `ultralytics`, `torch`, `numpy`, `Pillow`, `PyYAML`, `IPython` và có
dữ liệu/checkpoint tương ứng với các đường dẫn trong dự án.

In [ ]:
import csv
import hashlib
import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import yaml
from PIL import Image, ImageDraw
from ultralytics import YOLO
import ultralytics
from IPython.display import Markdown, display


ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "configs/threshold_yolov8s.yaml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Mở notebook từ thư mục dự án hoặc scripts/.")
OUT = ROOT / "evaluation/analysis"
CLASSES = ["Missing_hole", "Mouse_bite", "Open_circuit", "Short", "Spur", "Spurious_copper"]
CRITICAL = {2, 3}
MATCH_IOU = 0.5
PRED_CONF = 0.001
NMS_IOU = 0.7
IMGSZ = 1024
MAX_DET = 300
CONFIGS = ("B0", "B1", "B2", "Proposed")

## 1. Đọc nhãn và suy luận

In [ ]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def write_csv(path, rows, fields):
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    return inter / max(area_a + area_b - inter, 1e-12)


def label_path(image):
    parts = list(image.parts)
    parts[parts.index("images")] = "labels"
    return Path(*parts).with_suffix(".txt")


def load_gt(image):
    with Image.open(image) as im:
        width, height = im.size
    label = label_path(image)
    if not label.is_file():
        raise FileNotFoundError(label)
    gt = []
    for line in label.read_text(encoding="utf-8").splitlines():
        values = line.split()
        if len(values) < 5:
            continue  # Same rule as the locked comparison notebook.
        cid, xc, yc, w, h = map(float, values[:5])
        cid = int(cid)
        if cid not in range(6):
            continue
        area640 = w * h * 640**2
        size = "small" if area640 < 32**2 else "medium" if area640 <= 96**2 else "large"
        gt.append({"cid": cid, "box": [(xc-w/2)*width, (yc-h/2)*height,
                                        (xc+w/2)*width, (yc+h/2)*height], "size": size})
    return gt


def infer(checkpoint, images, device, batch):
    model = YOLO(str(checkpoint))
    # A txt source streams batches; a Python list can be interpreted as one huge batch.
    source = OUT / "test_image_paths.txt"
    predictions = []
    with torch.no_grad():
        for result in model.predict(source=str(source), imgsz=IMGSZ, batch=batch,
                                    device=device, conf=PRED_CONF, iou=NMS_IOU,
                                    max_det=MAX_DET, stream=True, verbose=False):
            predictions.append([] if result.boxes is None else [
                {"box": box.tolist(), "cid": int(cid), "score": float(score)}
                for box, cid, score in zip(result.boxes.xyxy.cpu().numpy(),
                                            result.boxes.cls.cpu().numpy(),
                                            result.boxes.conf.cpu().numpy())])
    if len(predictions) != len(images):
        raise RuntimeError(f"Inference returned {len(predictions)} results for {len(images)} images")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return predictions

## 2. Ghép dự đoán với GT và truy vết FP/FN

In [ ]:
def nearest(box, detections, same_class=None):
    pool = [d for d in detections if same_class is None or d["cid"] == same_class]
    if not pool:
        return None, 0.0
    return max(((d, iou(box, d["box"])) for d in pool), key=lambda x: x[1])


def explain_fn(g, all_dets, kept, matched_to):
    near, overlap = nearest(g["box"], all_dets, g["cid"])
    if near and overlap >= MATCH_IOU:
        if near not in kept:
            reason = "below_class_threshold"
        elif any(d is near for d in matched_to):
            reason = "candidate_assigned_to_other_gt"
        else:
            reason = "unmatched_after_greedy_matching"
    elif near and overlap > 0:
        reason = "localization_below_iou"
    else:
        other, other_iou = nearest(g["box"], kept)
        reason = "possible_class_confusion" if other and other_iou >= MATCH_IOU else "no_overlapping_detection"
    return reason, near, overlap


def evaluate(images, all_gt, predictions, model_name, name, tc, tr):
    details, matches = [], []
    gt_count, tp, fp = Counter(), Counter(), Counter()
    size_count, size_tp = Counter(), Counter()
    for image_index, (image, gt, dets) in enumerate(zip(images, all_gt, predictions)):
        for g in gt:
            gt_count[g["cid"]] += 1
            size_count[g["size"]] += 1
        kept = [d for d in dets if d["score"] >= (tc if d["cid"] in CRITICAL else tr)]
        assigned = {}
        matched_to = []
        for detection in sorted(kept, key=lambda d: -d["score"]):
            candidates = [(j, g) for j, g in enumerate(gt)
                          if g["cid"] == detection["cid"] and j not in assigned]
            if candidates:
                j, g = max(candidates, key=lambda item: iou(detection["box"], item[1]["box"]))
                if iou(detection["box"], g["box"]) >= MATCH_IOU:
                    assigned[j] = detection
                    matched_to.append(detection)
                    tp[g["cid"]] += 1
                    size_tp[g["size"]] += 1
                    continue
            fp[detection["cid"]] += 1
            other, overlap = nearest(detection["box"], gt)
            reason = ("possible_class_confusion" if other and overlap >= MATCH_IOU
                      and other["cid"] != detection["cid"] else
                      "possible_duplicate_or_competition" if other and overlap >= MATCH_IOU else
                      "localization_below_iou" if other and overlap > 0 else "no_overlapping_gt")
            details.append({"configuration": name, "model": model_name, "kind": "FP",
                            "image_index": image_index, "image": str(image.relative_to(ROOT)),
                            "item_index": kept.index(detection), "class": CLASSES[detection["cid"]],
                            "size_bin": "", "reason_candidate": reason,
                            "box_xyxy": json.dumps(detection["box"]),
                            "nearest_box_xyxy": json.dumps(other["box"]) if other else "",
                            "nearest_class": CLASSES[other["cid"]] if other else "",
                            "confidence": detection["score"], "nearest_iou": overlap,
                            "nearest_confidence": "", "gt_status_other_configs": ""})
        for j, g in enumerate(gt):
            matches.append({"configuration": name, "image_index": image_index,
                            "gt_index": j, "matched": j in assigned})
            if j in assigned:
                continue
            reason, near, overlap = explain_fn(g, dets, kept, matched_to)
            details.append({"configuration": name, "model": model_name, "kind": "FN",
                            "image_index": image_index, "image": str(image.relative_to(ROOT)),
                            "item_index": j, "class": CLASSES[g["cid"]], "size_bin": g["size"],
                            "reason_candidate": reason, "box_xyxy": json.dumps(g["box"]),
                            "nearest_box_xyxy": json.dumps(near["box"]) if near else "",
                            "nearest_class": CLASSES[near["cid"]] if near else "",
                            "confidence": "", "nearest_iou": overlap,
                            "nearest_confidence": near["score"] if near else "",
                            "gt_status_other_configs": ""})
    counts = {cid: {"GT": gt_count[cid], "TP": tp[cid], "FP": fp[cid],
                    "FN": gt_count[cid]-tp[cid]} for cid in range(6)}
    sizes = {size: {"GT": size_count[size], "TP": size_tp[size],
                    "FN": size_count[size]-size_tp[size]}
             for size in ("small", "medium", "large")}
    return details, matches, counts, sizes


def comparison_rows(results, key, order):
    rows = []
    for category in order:
        row = {key: CLASSES[category] if key == "class" else category}
        for name in CONFIGS:
            counts = results[name]["counts"] if key == "class" else results[name]["sizes"]
            data = counts[category]
            for field in ("GT", "TP", "FN") + (("FP",) if key == "class" else ()):
                row[f"{name}_{field}"] = data[field]
            row[f"{name}_recall"] = data["TP"] / data["GT"] if data["GT"] else ""
            if key == "class":
                row[f"{name}_precision"] = (data["TP"] / (data["TP"]+data["FP"])
                                               if data["TP"]+data["FP"] else 0.0)
        for first, second in (("B0", "B1"), ("B0", "B2"), ("B1", "Proposed")):
            a, b = row[f"{first}_recall"], row[f"{second}_recall"]
            row[f"delta_recall_{second}_minus_{first}"] = b-a if a != "" and b != "" else ""
        rows.append(row)
    return rows

## 3. Minh họa lỗi và viết báo cáo

In [ ]:
def render_gallery(images, all_gt, predictions, details, status, limit):
    gallery = OUT / "fp_fn_gallery"
    gallery.mkdir(exist_ok=True)
    # Prefer distinct images for each case, deterministically in split order.
    selections = (("b0_tp_b1_fn", "B1", "FN", lambda s: s["B0"] and not s["B1"]),
                  ("b1_tp_proposed_fn", "Proposed", "FN", lambda s: s["B1"] and not s["Proposed"]),
                  ("proposed_short_fp", "Proposed", "FP", lambda s: True))
    shown = []
    for category, configuration, kind, eligible in selections:
        used_images = set()
        used_origins = set()
        candidates = [d for d in details if d["configuration"] == configuration and d["kind"] == kind
                      and (kind != "FP" or d["class"] == "Short" and float(d["confidence"]) < 0.25)
                      and eligible(status.get((d["image_index"], d["item_index"]), {}))]
        if kind == "FP":
            candidates.sort(key=lambda d: (d["reason_candidate"] != "no_overlapping_gt",
                                           -float(d["confidence"]), d["image_index"]))
        for d in candidates:
            origin = re.sub(r"_[123]$", "", images[d["image_index"]].stem)
            if d["image_index"] in used_images or origin in used_origins:
                continue
            used_images.add(d["image_index"])
            used_origins.add(origin)
            index = d["image_index"]
            model_name = "B0" if configuration in ("B0", "B2") else "B1"
            tc, tr = (0.25, 0.25) if configuration in ("B0", "B1") else THRESHOLDS[model_name]
            kept = [p for p in predictions[model_name][index]
                    if p["score"] >= (tc if p["cid"] in CRITICAL else tr)]
            with Image.open(images[index]) as original:
                canvas = original.convert("RGB")
            draw = ImageDraw.Draw(canvas)
            for g in all_gt[index]:
                draw.rectangle(g["box"], outline="#00ff66", width=3)
            for p in kept:
                draw.rectangle(p["box"], outline="#ff3344", width=2)
            focus = json.loads(d["box_xyxy"])
            draw.rectangle(focus, outline="#ffff00", width=5)
            # Crop around the error; retain nearby context and cap output size.
            x1, y1, x2, y2 = focus
            margin = max(120, (x2-x1)*2, (y2-y1)*2)
            crop = (max(0, int(x1-margin)), max(0, int(y1-margin)),
                    min(canvas.width, int(x2+margin)), min(canvas.height, int(y2+margin)))
            name = f"{category}_{len(used_images):02d}_{images[index].stem}.jpg"
            canvas.crop(crop).save(gallery / name, quality=88)
            shown.append((name, d, category))
            if len(used_images) >= limit:
                break
    selected = {name for name, _, _ in shown}
    for stale in gallery.glob("*.jpg"):
        if stale.name not in selected and stale.name.startswith(
                ("b0_tp_b1_fn_", "b1_tp_proposed_fn_", "proposed_short_fp_")):
            stale.unlink()
    return shown


def report(rows_class, rows_size, details, gallery, checks, hardware, differences, reasons, primary):
    def frac(value):
        return "N/A" if value == "" else f"{float(value):.4f}"

    lines = ["# Phân tích lỗi YOLOv8s và YOLOv8s-P2", "",
             "## Giao thức và cách tái lập", "",
             "- B0: YOLOv8s, conf=0.25; B1: YOLOv8s-P2, conf=0.25. B2 dùng B0 với "
             "(critical=0.02, regular=0.58); Proposed dùng B1 với (critical=0.01, regular=0.57).",
             "- 480 ảnh test cố định; 2.424 GT; imgsz=1024, inference conf=0.001, "
             "NMS IoU=0.7, max_det=300; matching theo lớp, một-một, greedy theo confidence, IoU>=0.5.",
             "- Size bin từ box GT: area640=w_norm × h_norm × 640²; small < 32², "
             "medium <= 96², large > 96². Đây không phải kích thước pixel ảnh gốc hoặc imgsz=1024.",
             f"- Thiết bị tái chạy: {hardware}; ultralytics=={ultralytics.__version__}; "
             f"test split SHA256 `{checks['test_split_sha256']}`. "
             "Checkpoint được kiểm tra SHA256 với cấu hình threshold đã khóa.",
             "- Bảng tuần 10 được tạo trên Kaggle CUDA (notebook cài ultralytics==8.3.40). "
             "Hai bảng dưới đây là số liệu *gốc đã khóa*; ảnh và `detection_errors.csv` "
             "là kết quả tái chạy ở môi trường hiện tại. Bảng tái chạy nằm ở "
             "`per_class_rerun.csv` và `size_bin_rerun.csv`.",
             "- Chạy lại: mở `scripts/analyze_detection_errors.ipynb` và chọn Run All "
             "(cần ultralytics, torch, numpy, Pillow, PyYAML và các ảnh/checkpoint hiện có).", "",
             "## Kết quả theo kích thước", "",
             "| Bin | GT | B0 Recall | B1 Recall | B2 Recall | Proposed Recall |", "|---|---:|---:|---:|---:|---:|"]
    for row in rows_size:
        lines.append(f"| {row['size_bin']} | {row['B0_GT']} | " + " | ".join(
            frac(row[f"{c}_recall"]) for c in CONFIGS) + " |")
    lines += ["", "## Kết quả theo lớp", "",
              "| Lớp | GT | B0 TP/FP/FN | B1 TP/FP/FN | B2 TP/FP/FN | Proposed TP/FP/FN |",
              "|---|---:|---:|---:|---:|---:|"]
    for row in rows_class:
        vals = [f"{row[f'{c}_TP']}/{row[f'{c}_FP']}/{row[f'{c}_FN']}" for c in CONFIGS]
        lines.append(f"| {row['class']} | {row['B0_GT']} | " + " | ".join(vals) + " |")
    if differences:
        lines += ["", "**Sai khác tái lập:** Có " + str(len(differences)) +
                  " ô đếm GT/TP/FP/FN khác kết quả tuần 10 (chi tiết tại `metric_reproduction_differences.csv`). "
                  "Không dùng số tái chạy để thay thế chỉ số gốc; ảnh lỗi chỉ là ví dụ "
                  "theo suy luận ở môi trường hiện tại. Khác biệt môi trường là một khả năng, "
                  "chưa được xác nhận là nguyên nhân duy nhất."]
    else:
        lines += ["", "Tất cả ô đếm GT/TP/FP/FN khớp bảng tuần 10."]
    p = {r["configuration"]: r for r in primary}
    lines += ["", "## Phân tích", "",
              f"- B0→B1: recall small {frac(rows_size[0]['B0_recall'])} → "
              f"{frac(rows_size[0]['B1_recall'])}, medium {frac(rows_size[1]['B0_recall'])} → "
              f"{frac(rows_size[1]['B1_recall'])}; mAP50-95 "
              f"{float(p['B0']['mAP50_95']):.4f} → {float(p['B1']['mAP50_95']):.4f}. "
              "Trong test này P2 không cải thiện nhất quán lỗi nhỏ. Missing_hole giảm "
              "38 TP (383→345) và tăng 38 FN (25→63) ở ngưỡng 0.25.",
              f"- B0→B2: critical recall {float(p['B0']['critical_recall']):.4f} → "
              f"{float(p['B2']['critical_recall']):.4f}, nhưng F2 "
              f"{float(p['B0']['F2']):.4f} → {float(p['B2']['F2']):.4f}. "
              "Open_circuit FP tăng 32→418; Spur FN tăng 27→61.",
              f"- B1→Proposed: Short FP tăng 295→692 trong khi Short TP chỉ tăng 374→377; "
              f"F2 {float(p['B1']['F2']):.4f} → {float(p['Proposed']['F2']):.4f}. "
              "Ngưỡng thấp của lớp critical làm tăng detection sai, ngưỡng regular cao "
              "loại bớt các TP có confidence thấp. Xem ví dụ cụ thể ở gallery.",
              "- GT large bằng 0 trên test đang đánh giá nên không thể kết luận về hiệu quả trên large. "
              "Không dùng thống kê từ một split/định nghĩa kích thước khác để điền giá trị.",
              "", "## Nhóm dấu hiệu trên kết quả tái chạy", "",
              "Các đếm sau đây là từ MPS; đối chiếu sai khác với số liệu gốc trước khi trích dẫn.", "",
              "| Cấu hình | Loại | Dấu hiệu gợi ý | Số trường hợp |", "|---|---|---|---:|"]
    for r in reasons:
        lines.append(f"| {r['configuration']} | {r['kind']} | {r['reason_candidate']} | {r['count']} |")
    lines += ["",
              "- `reason_candidate` chỉ là gợi ý dựa trên dự đoán *sau* NMS. "
              "Không thể xác nhận box bị NMS loại hay chất lượng nhãn chỉ từ đầu ra này; "
              "muốn kết luận về NMS phải thu thập dự đoán trước NMS.",
              "- So sánh B0/B1 không cô lập hoàn toàn tác động P2 vì hai checkpoint đã được "
              "huấn luyện riêng (B0 batch 16, B1 batch 8; một seed).", "",
              "## Ví dụ FP/FN", "",
              "Màu xanh lá: GT; đỏ: dự đoán vượt ngưỡng; vàng: box đang xem. "
              "Ảnh được crop để quan sát; CSV có đường dẫn ảnh gốc, box, IoU và confidence.", ""]
    for name, d, category in gallery:
        lines += [f"- [{category} — {d['class']}, {d['reason_candidate']}; "
                  f"ảnh `{Path(d['image']).name}`](fp_fn_gallery/{name}) "
                  f"(IoU gần nhất {float(d['nearest_iou']):.3f}, "
                  f"confidence gần nhất {float(d['nearest_confidence']):.3f})." if d["kind"] == "FN" else
                  f"- [{category} — {d['class']}, {d['reason_candidate']}; "
                  f"ảnh `{Path(d['image']).name}`](fp_fn_gallery/{name}) "
                  f"(confidence {float(d['confidence']):.3f}, IoU gần nhất {float(d['nearest_iou']):.3f})."]
    if not gallery:
        lines.append("- Không tìm thấy ví dụ thuộc các nhóm lựa chọn.")
    lines += ["", "## Giới hạn và liên hệ đề cương", "",
              "- Toàn bộ ảnh test đều có lỗi; tỷ lệ ảnh sạch → DEFECTIVE không tính được. "
              "Không suy diễn đây là 0% false alarm trên bo mạch sạch.",
              "- B1 thực tế là YOLOv8s-P2 (checkpoint riêng), không phải SAHI/tiling cùng checkpoint "
              "như đề cương ban đầu. Sửa phần phương pháp/câu hỏi nghiên cứu để nhất quán; "
              "không gán suy giảm cho patch boundary khi không có patch.",
              "- Test chỉ dùng mô tả/giải thích kết quả. Không điều chỉnh ngưỡng, NMS, "
              "checkpoint hoặc chọn lại cấu hình theo các ví dụ trong gallery.", ""]
    (OUT / "error_analysis_report.md").write_text("\n".join(lines), encoding="utf-8")

## 4. Chạy phân tích trên test split đã khóa

Có thể đổi `device` (`None` = tự chọn CUDA/MPS/CPU), `batch` và số ảnh
minh họa mỗi nhóm trong ô cuối. Không thay đổi split/threshold theo kết quả test.

In [ ]:
def main(device=None, batch=2, gallery_per_category=4):
    if batch < 1 or gallery_per_category < 1:
        raise ValueError("batch và gallery_per_category phải lớn hơn 0")
    split = ROOT / "data/splits/test.txt"
    protocol = json.loads((ROOT / "evaluation/B0_B1_yolov8s/week10_protocol_and_metrics.json").read_text())
    config = yaml.safe_load((ROOT / "configs/threshold_yolov8s.yaml").read_text())
    if sha256(split) != protocol["test_split_sha256"]:
        raise RuntimeError("Test split does not match the locked comparison")
    # Keep the repository-relative spelling: raw images can be symlinked to an external dataset.
    images = [ROOT / line for line in split.read_text().splitlines() if line.strip()]
    if len(images) != protocol["test_images"] or not all(i.is_file() for i in images):
        raise RuntimeError("Test image list differs from the locked comparison")
    checkpoints = {}
    global THRESHOLDS
    THRESHOLDS = {}
    for name in ("B0", "B1"):
        info = config["models"][name]
        path = ROOT / info["checkpoint"]
        if not path.is_file() or sha256(path) != protocol["checkpoint_sha256"][name] or sha256(path) != info["checkpoint_sha256"]:
            raise RuntimeError(f"Checkpoint mismatch: {name}")
        checkpoints[name] = path
        THRESHOLDS[name] = (info["tau_critical"], info["tau_regular"])
    gt = [load_gt(image) for image in images]
    if sum(map(len, gt)) != 2424:
        raise RuntimeError("GT count differs from the locked comparison")
    OUT.mkdir(parents=True, exist_ok=True)
    (OUT / "test_image_paths.txt").write_text("".join(f"{p}\n" for p in images), encoding="utf-8")
    device = device or ("0" if torch.cuda.is_available() else
                        "mps" if torch.backends.mps.is_available() else "cpu")
    predictions = {}
    for name, checkpoint in checkpoints.items():
        print(f"Inferring {name} on {len(images)} images ({device})...", flush=True)
        predictions[name] = infer(checkpoint, images, device, batch)
    specs = (("B0", "B0", .25, .25), ("B1", "B1", .25, .25),
             ("B2", "B0", *THRESHOLDS["B0"]), ("Proposed", "B1", *THRESHOLDS["B1"]))
    results, details, matches = {}, [], []
    for name, model, tc, tr in specs:
        items, assigned, counts, sizes = evaluate(images, gt, predictions[model], model, name, tc, tr)
        results[name] = {"counts": counts, "sizes": sizes}
        details.extend(items)
        matches.extend(assigned)
    with (ROOT / "evaluation/B0_B1_yolov8s/week10_per_class_metrics.csv").open(newline="", encoding="utf-8") as f:
        original = list(csv.DictReader(f))
    differences = []
    for item in original:
        c = results[item["configuration"]]["counts"][CLASSES.index(item["class"])]
        for field in ("GT", "TP", "FP", "FN"):
            if c[field] != int(item[field]):
                differences.append({"configuration": item["configuration"], "class": item["class"],
                                    "metric": field, "locked": item[field], "rerun": c[field],
                                    "delta": c[field]-int(item[field])})
    status = {}
    for item in matches:
        status.setdefault((item["image_index"], item["gt_index"]), {})[item["configuration"]] = item["matched"]
    for item in details:
        if item["kind"] == "FN":
            item["gt_status_other_configs"] = json.dumps(status[item["image_index"], item["item_index"]])
    rerun_class = comparison_rows(results, "class", range(6))
    rerun_size = comparison_rows(results, "size_bin", ("small", "medium", "large"))
    locked = {name: {"counts": {}, "sizes": {}} for name in CONFIGS}
    for item in original:
        locked[item["configuration"]]["counts"][CLASSES.index(item["class"])] = {
            field: int(item[field]) for field in ("GT", "TP", "FP", "FN")}
    with (ROOT / "evaluation/B0_B1_yolov8s/week10_primary_metrics.csv").open(newline="", encoding="utf-8") as f:
        primary = list(csv.DictReader(f))
    for item in primary:
        name = item["configuration"]
        for size in ("small", "medium", "large"):
            n = results["B0"]["sizes"][size]["GT"]
            metric = item[f"R_{size}"]
            if (not n) != (metric == ""):
                raise RuntimeError(f"Locked size bin {size} does not match GT support")
            hits = round(n * float(metric)) if metric else 0
            if metric and abs(hits/n - float(metric)) > 1e-10:
                raise RuntimeError(f"Locked recall for {name} {size} cannot map to integer TP")
            locked[name]["sizes"][size] = {"GT": n, "TP": hits, "FN": n-hits}
    per_class = comparison_rows(locked, "class", range(6))
    per_size = comparison_rows(locked, "size_bin", ("small", "medium", "large"))
    write_csv(OUT / "per_class_comparison.csv", per_class, list(per_class[0]))
    write_csv(OUT / "size_bin_comparison.csv", per_size, list(per_size[0]))
    write_csv(OUT / "per_class_rerun.csv", rerun_class, list(rerun_class[0]))
    write_csv(OUT / "size_bin_rerun.csv", rerun_size, list(rerun_size[0]))
    write_csv(OUT / "detection_errors.csv", details, list(details[0]))
    write_csv(OUT / "metric_reproduction_differences.csv", differences,
              ["configuration", "class", "metric", "locked", "rerun", "delta"])
    reason_counts = Counter((d["configuration"], d["kind"], d["reason_candidate"]) for d in details)
    reasons = [{"configuration": name, "kind": kind, "reason_candidate": reason, "count": count}
               for (name, kind, reason), count in sorted(reason_counts.items())]
    write_csv(OUT / "error_reason_summary.csv", reasons,
              ["configuration", "kind", "reason_candidate", "count"])
    gallery = render_gallery(images, gt, predictions, details, status, gallery_per_category)
    report(per_class, per_size, details, gallery, protocol, device, differences, reasons, primary)
    print("GT size counts:", results["B0"]["sizes"])
    print("FP/FN rows:", len(details), "gallery images:", len(gallery))
    print("Differences versus locked per-class counts:", len(differences), "output:", OUT)

In [ ]:
main(device=None, batch=2, gallery_per_category=4)
display(Markdown((OUT / "error_analysis_report.md").read_text(encoding="utf-8")))